# Transformer LM
transformer_lm.py

**element-wise multiplies:**

**dot product:**

`Num of matrix multiplies: num_layers * 9 matrix multiplies + 1`

`FLOPs: (num_layers * (((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model)))) + (batch...) * (2 * seq_len * d_model * vocab_size) FLOPs`

* init: `0 matrix multiplies` `0 FLOPs`
* forward:
    * Num of matrix multiplies: `num_layers * 9 matrix multiplies + 1`
    * FLOPs: `(num_layers * (((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model)))) + (batch...) * (2 * seq_len * d_model * vocab_size) FLOPs`


**trainable parameters:** `vocab_size * d_model + num_layers * (d_model + d_model + 4 * (d_model * d_model) + 3 * (d_ff * d_model)) + d_model + vocab_size * d_model`
* init:
  * self.embedding_layer: Parameters: vocab_size * d_model
  * self.transformerblock_layer: Parameters: num_layers * (d_model + d_model + 4 * (d_model * d_model) + 3 * (d_ff * d_model))
  * self.rmsnorm_layer: Parameters: d_model
  * self.linear_layer: Parameters: vocab_size * d_model

* forward: 0 parameters


In [ ]:
import torch.nn as nn
from jaxtyping import Float, Int
from torch import Tensor
from cs336_basics.embedding import Embedding
from cs336_basics.transformer_block import TransformerBlock
from cs336_basics.rmsnorm_einx import RMSNorm
from cs336_basics.linear_module import Linear

class TransformerLM(nn.Module):
    def __init__(self, vocab_size: int, context_length: int, num_layers: int, d_model: int, num_heads: int, d_ff: int, rope_theta: float):
        """Implement the Transformer language model

            Args:
                vocab_size (int): The size of the vocabulary, necessary for determining the dimensionality of the token embedding matrix
                context_length (int): The maximum context length, necessary for determining the dimensionality of the position embedding matrix
                num_layers (int): The number of Transformer blocks to use
                d_model (int): Dimensionality of the Transformer block inputs
                num_heads (int): Number of heads to use in multi-head self-attention
                d_ff (int): Dimensionality of the position-wise feed-forward inner layer
                rope_theta (float): RoPE parameter
        """
        super().__init__()

        self.embedding_layer = Embedding(vocab_size, d_model) # 0 matrix multiplies; 0 FLOPs | Parameters: vocab_size * d_model
        self.transformerblock_layer = nn.ModuleList(
            [TransformerBlock(d_model, num_heads, d_ff, context_length, rope_theta) for _ in range(num_layers)]
        ) # num_layers * (0 matrix multiplies; 0 FLOPs) | Parameters: num_layers * (d_model + d_model + 4 * (d_model * d_model) + 3 * (d_ff * d_model))
        # self.transformerblock_layer = [TransformerBlock(d_model, num_heads, d_ff, context_length, rope_theta) for _ in range(num_layers)]
        self.rmsnorm_layer = RMSNorm(d_model) # 0 matrix multiplies; 0 FLOPs | Parameters: d_model
        self.linear_layer = Linear(d_model, vocab_size) # 0 matrix multiplies; 0 FLOPs | Parameters: vocab_size * d_model
    def forward(self, in_indices: Int[Tensor, " batch_size sequence_length"]) -> Float[Tensor, " batch_size sequence_length vocab_size"]:
        """Implement the Transformer language model

            Args:
                in_indices (Int[Tensor, " batch_size sequence_length"]): Tensor with input indices to run the language model on. Shape is (batch_size, sequence_length), where
            `sequence_length` is at most `context_length`

            Returns:
                Float[Tensor, "batch_size sequence_length vocab_size"]: Tensor with the predicted unnormalized next-word distribution for each token.
        """
        input_embedding = self.embedding_layer.forward(in_indices) # 0 matrix multiplies; 0 FLOPs | Parameters: 0 parameters

        for block in self.transformerblock_layer:                  # num_layers * (9 matrix multiplies; ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs) | Parameters: num_layers * 0 parameters
            output_embedding = block.forward(input_embedding)
            input_embedding = output_embedding

        output_embedding = self.rmsnorm_layer.forward(output_embedding) # 0 matrix multiplies; 0 FLOPs | Parameters: 0 parameters

        result = self.linear_layer.forward(output_embedding) # 1 matrix multiplies; FLOPs = (batch...) * (2 * seq_len * d_model * vocab_size) FLOPs | Parameters: 0 parameters

        return result

## TokenEmbedding

embedding.py

**dot product:**

* Num of matrix multiplies: `0 matrix multiplies`

* FLOPs: `0 FLOPs`

**trainable parameters:** vocab_size * d_model
* init:
  * self.W(vocab_size, d_model)
* forward: 0 parameters

In [ ]:
import torch
import torch.nn as nn

class Embedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """Construct an embedding module.

        Args:
            num_embeddings(int): Size of the vocabulary
            embedding_dim(int): Dimension of the embedding vectors
            device(torch.device | None): Device to store the parameters on
            dtype(torch.dtype | None): Data type of the parameters
        """
        super().__init__()
        temp_W = torch.empty(num_embeddings, embedding_dim, device=device, dtype=dtype)
        temp_W = torch.nn.init.trunc_normal_(temp_W, mean=0, std=1, a=-3, b=3)
        self.W = nn.Parameter(temp_W) # Parameter: self.W(vocab_size, d_model)
    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """Lookup the embedding vectors for the given token IDs.

        Args:
            token_ids(torch.Tensor): token ids with shape (batch_size, sequence_length)
        """
        return self.W[token_ids]

## Transformer Block
transformer_block.py

**dot product:**

`Num of matrix multiplies: 9 matrix multiplies`;

`FLOPs: ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs`

* init: `0 matrix multiplies`; `0 FLOPs`
* forward: `9 matrix multiplies`;`((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs`

**trainable parameters:** `d_model + d_model + 4 * (d_model * d_model) + 3 * (d_ff * d_model)`
* init:
  * self.rmsnorm_first_layer(Parameter: (d_model))
  * self.rmsnorm_second_layer(Parameter: (d_model))
  * self.multiheadselfattentionrops_layer(Parameter: 4 * (d_model * d_model))
  * self.positionwiseffn_layer(Parameter: 3 * (d_ff * d_model))
* forward: 0 parameters






In [ ]:
import torch
import torch.nn as nn
from jaxtyping import Float
from torch import Tensor
from cs336_basics.rmsnorm_einx import RMSNorm
from cs336_basics.multihead_self_attention_rope import MultiHeadSelfAttentionRope
from cs336_basics.positionwise_feedforward_einx import PWFFN

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, max_seq_len: int, theta: float):
        """Implement the pre-norm Transformer block

            Args:
                d_model (int): Dimensionality of the Transformer block inputs
                num_heads (int): Number of heads to use in multi-head self-attention
                d_ff (int): Dimensionality of the position-wise feed-forward inner layer
                max_seq_len (int): Maximum sequence length to pre-cache if your implementation does that.
                theta (float): RoPE parameter
        """
        super().__init__()

        self.rmsnorm_first_layer = RMSNorm(d_model) # 0 matrix multiplies; 0 FLOPs | Parameter: (d_model)
        self.rmsnorm_second_layer = RMSNorm(d_model) # 0 matrix multiplies; 0 FLOPs | Parameter: (d_model)
        self.multiheadselfattentionrops_layer = MultiHeadSelfAttentionRope(d_model, num_heads, max_seq_len, theta) # 0 matrix multiplies, 0 FLOPs | Parameter: 4 * (d_model * d_model)
        self.positionwiseffn_layer = PWFFN(d_model, d_ff) # 0 matrix multiplies; 0 FLOPs | Parameter: 3 * (d_ff * d_model)
    def forward(self, x: Float[Tensor, " batch sequence_length d_model"]) -> Float[Tensor, " batch sequence_length d_model"]:
        x_norm = self.rmsnorm_first_layer.forward(x) # 0 matrix multiplies; 0 FLOPs | Parameter: 0 parameters
        # embedding_attention = self.multiheadselfattentionrops_layer.forward(x_norm)
        x_seq_len = x.size(-2)
        token_positions = torch.arange(x_seq_len).unsqueeze(0).expand(*x.shape[:-2], x_seq_len)
        embedding_attention = self.multiheadselfattentionrops_layer.forward(x_norm, token_positions) # 6 matrix multiplies; FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) FLOPs |Parameters: 0 parameters

        result_firstsublayer = x + embedding_attention

        result_firstsublayer_norm = self.rmsnorm_second_layer.forward(result_firstsublayer) # 0 matrix multiplies; 0 FLOPs | Parameter: 0 parameters
        embedding_pwffn = self.positionwiseffn_layer.forward(result_firstsublayer_norm) # 3 matrix multiplies; (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs | Parameter: 0 parameters
        result_secondsublayer = result_firstsublayer + embedding_pwffn

        return result_secondsublayer

### Norm
rmsnorm_einx.py

**element-wise multiplies:**

**dot product:**

`0 matrix multiplies`

`0 FLOPs`

**trainable parameters:** `d_model`
* init:
  * self.g(d_model)
* forward: 0 parameters

In [ ]:
import torch
import torch.nn as nn
import einx

class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """ Construct the RMSNorm module.

        Args:
            d_model(int): Hidden dimension of the model
            eps(float): Epsilon value for numerical stability
            device(torch.device | None = None): Device to store the parameters on
            dtype(torch.dtype | None = None): Data type of the parameters
        """
        super().__init__()
        self.g = nn.Parameter(torch.ones(d_model, dtype=dtype, device=device)) # Parameter: self.g(d_model)
        self.eps = eps
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Process an input tensor of shape

        Args:
            x (torch.Tensor): shape (batch_size, sequence_length, d_model)
        Returns:
            torch.Tensor: shape (batch_size, sequence_length, d_model)
        """
        in_dtype = x.dtype
        x = x.to(torch.float32)

        mean_square = einx.mean('... d -> ... 1', x * x)
        inv_rms = torch.rsqrt(mean_square + self.eps)

        x_norm = einx.multiply('... d, ... 1 -> ... d', x, inv_rms) # Activations: x(batch_size..., seq_len, d_model)
        result = einx.multiply('... d, d -> ... d', x_norm, self.g) # Activations: x_norm(batch_size..., seq_len, d_model)

        return result.to(in_dtype)


### Causal Multi-Head Self-Attention w/ RoPE
multihead_self_attention_rope.py

**element-wise multiplies:**

**dot product:**

`6 matrix multiplies`;

`((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) FLOPs`
  * init: `0 matrix multiplies`, `0 FLOPs`
  * forward:
      * `6 matrix multiplies`
      * `FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) FLOPs`

**trainable parameters:** `4 * (d_model * d_model)`
* init:
  * self.Q(d_model, d_model)
  * self.K(d_model, d_model)
  * self.V(d_model, d_model)
  * self.O(d_model, d_model)
* forward: 0 parameters

In [ ]:
import torch
import torch.nn as nn
from jaxtyping import Float, Int
from torch import Tensor
from cs336_basics.scaled_dot_product_attention import SDPAttention
from cs336_basics.rope_einx import RoPe

class MultiHeadSelfAttentionRope(nn.Module):
    def __init__(self, d_model: int, num_heads: int, max_seq_len: int, theta: float):
        """Causal multi-head self-attention

            Args:
                d_model (int): Dimensionality of the Transformer block inputs
                num_heads (int): Number of heads to use in multi-head self-attention
                max_seq_len (int): Maximum sequence length to pre-cache
                theta (float): RoPE parameter
        """
        super().__init__()

        self.Q = nn.Parameter(torch.randn(d_model, d_model)) # Parameter: self.Q(d_model, d_model)
        self.K = nn.Parameter(torch.randn(d_model, d_model)) # Parameter: self.K(d_model, d_model)
        self.V = nn.Parameter(torch.randn(d_model, d_model)) # Parameter: self.V(d_model, d_model)
        self.O = nn.Parameter(torch.randn(d_model, d_model)) # Parameter: self.O(d_model, d_model)

        self.num_heads = num_heads

        self.rope_layer = RoPe(theta=theta, d_k=d_model // num_heads, max_seq_len=max_seq_len) # 0 matrix multiplies, 0 FLOPs | Parameter: 0 parameters

    def forward(self, x: Float[Tensor, " ... sequence_length d_in"], token_positions: Int[Tensor, " ... sequence_length"] | None = None) -> Float[Tensor, " ... sequence_length d_out"]:
        batch_shape = x.shape[:-2]
        seq_len = x.shape[-2]

        q_x = x @ self.Q.T  # x(... sequence_length d_model), self.Q.T(d_model, d_model); FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model); # Activations: self.Q.T[para-ignore](d_model, d_model), x(batch_size..., seq_len, d_model)
        k_x = x @ self.K.T  # x(... sequence_length d_model), self.K.T(d_model, d_model); FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model); # Activations: self.K.T[para-ignore](d_model, d_model), x[repeat-ignore](batch_size..., seq_len, d_model)
        v_x = x @ self.V.T  # x(... sequence_length d_model), self.V.T(d_model, d_model); FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model); # Activations: self.V.T[para-ignore](d_model, d_model), x[repeat-ignore](batch_size..., seq_len, d_model)

        q_x_heads = q_x.reshape(*batch_shape, seq_len, self.num_heads, -1)
        k_x_heads = k_x.reshape(*batch_shape, seq_len, self.num_heads, -1)
        v_x_heads = v_x.reshape(*batch_shape, seq_len, self.num_heads, -1)

        q_x_heads = q_x_heads.transpose(-3, 2)
        k_x_heads = k_x_heads.transpose(-3, 2)
        v_x_heads = v_x_heads.transpose(-3, 2)

        q_x_heads = self.rope_layer.forward(q_x_heads, token_positions) # 0 matrix multiplies, 0 FLOPs
        k_x_heads = self.rope_layer.forward(k_x_heads, token_positions) # 0 matrix multiplies, 0 FLOPs

        causal_mask = ~torch.triu(torch.ones(x.shape[-2], x.shape[-2], dtype=bool), diagonal=1)

        attention_layer = SDPAttention(q_x_heads, k_x_heads, v_x_heads, causal_mask) # 0 matrix multiplies; 0 FLOPs | Parameter: 0 parameters
        embedding_cmhsa = attention_layer.forward() # 2 matrix multiplies; (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) FLOPs | Parameter: 0 parameters
        embedding_cmhsa_trans = embedding_cmhsa.transpose(-3, -2)
        embedding_cmhsa_combined = embedding_cmhsa_trans.contiguous().reshape(*batch_shape, seq_len, -1)
        result = embedding_cmhsa_combined @ self.O.T # embedding_cmhsa_combined(batch..., seq_len, d_model), self.O.T(d_model, d_model); FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model); # Activations: self.O.T[para-ignore](d_model, d_model), embedding_cmhsa_combined(batch_size..., seq_len ,d_model)

        return result

#### Scaled Dot-Product Attention
scaled_dot_product_attention.py

**element-wise multiplies:**

**dot product:**
  * init: `0 matrix multiplies`, `0 FLOPs`
  * forward:
      * `2 matrix multiplies`
      * `(batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) FLOPs`

**trainable parameters:** `0 parameters`

In [ ]:
import torch
import torch.nn as nn
from jaxtyping import Float
from torch import Tensor
import cs336_basics.softmax_einx as sm
import math

class SDPAttention(nn.Module):
    def __init__(self, q: Float[Tensor, "... seq_len d_k"], k: Float[Tensor, "... seq_len d_k"], v: Float[Tensor, "... seq_len d_v"], mask: Float[Tensor, " ... queries keys"] | None = None):
        super().__init__()

        self.Q = q
        self.K = k
        self.V = v
        self.mask = mask
    def forward(self) -> Float[Tensor, "... d_v"]:
        qk = self.Q @ self.K.transpose(-2, -1) # self.Q(... num_heads, seq_len d_model), self.K.transpose(-2, -1)(... num_heads, d_model seq_len); FLOPs = (batch...) * (2 * seq_len * d_model * seq_len); # Activations: self.K.transpose(-2, -1)(batch_size..., num_heads, d_model, seq_len), self.Q(batch_size..., num_heads, seq_len, d_model)
        qk_norm = qk / math.sqrt(self.Q.size(-1))

        mask_ninf = torch.where(self.mask, torch.zeros_like(self.mask), float('-inf'))

        qk_norm_mask = qk_norm + mask_ninf

        attention_score = sm.Softmax(qk_norm_mask, -1) # 0 matrix multiplies, 0 FLOPs | Parameter: 0 parameters; # Activations: attention_score(batch_size..., num_heads, seq_len, seq_len)
        result = attention_score @ self.V # attention_score(... seq_len seq_len), self.V(... seq_len d_model); FLOPs = (batch...) * (2 * seq_len * seq_len * d_model); # Activations: attention_score[repeat-ignore](batch_size..., num_heads, seq_len, seq_len), self.V(batch_size..., num_heads, seq_len, d_model)

        return result



##### Softmax
softmax_einx.py

**element-wise multiplies:**

**dot product:**

`0 matrix multiplies`,

`0 FLOPs`

**trainable parameters:** `0 parameters`

In [ ]:
import torch
from jaxtyping import Float
from torch import Tensor
import einx

def Softmax(x: Float[Tensor, " ..."], dim: int) -> Float[Tensor, " ..."]:
    logit_stable = einx.subtract("... logits, ... 1 -> ... logits", x, x.max(dim=dim, keepdim=True).values)
    logit_stable_exp = torch.exp(logit_stable)
    result = einx.divide("... logits, ... 1 -> ... logits", logit_stable_exp, logit_stable_exp.sum(dim=dim, keepdim=True))

    return result


#### RoPE
rope_einx.py

**element-wise multiplies:**
**dot product:**

`Num matrix multiplies: 0 matrix multiplies`

`FLOPs: 0 FLOPs`

  * init: `0 matrix multiplies` `0 FLOPs`
  * forward: `0 matrix multiplies` `0 FLOPs`

**trainable parameters:**
`0 parameters`

In [ ]:
import torch
import torch.nn as nn
from jaxtyping import Float, Int
from torch import Tensor
import einx

class RoPe(nn.Module):
    def __init__(self, theta: float, d_k: int, max_seq_len: int, device: torch.device | None = None):
        """Constructthe RoPE module and create buffers if needed.

        Args:
            theta (float): Θ value for the RoPE
            d_k (int): dimension of query and key vectors
            max_seq_len (int): Maximum sequence length that will be inputted
            device (torch.device | None): Device to store the buffer on
        """
        super().__init__()

        block_num = d_k // 2

        angle_i = torch.arange(max_seq_len)
        angle_k = torch.arange(1, block_num + 1)
        angle = einx.multiply("max_seq_len 1, 1 block_num -> max_seq_len block_num", angle_i[:, None], torch.reciprocal(theta ** ((2*angle_k[None, :] - 2) / d_k)))

        sin = torch.sin(angle)
        cos = torch.cos(angle)

        self.register_buffer("sin", sin, persistent=False)
        self.register_buffer("cos", cos, persistent=False)

    def forward(self, x: Float[Tensor, "... seq_len d_k"], token_positions: Int[Tensor, "... seq_len"]) -> Float[Tensor, "... seq_len d_k"]:
        *batch, seq_len, d_k = x.shape
        block = d_k // 2

        x_blocked = x.reshape(*batch, seq_len, block, -1)

        x_even = x_blocked[..., 0]
        x_odd = x_blocked[..., 1]

        sin_pos = self.sin[token_positions]
        cos_pos = self.cos[token_positions]

        x_even_rot = x_even * cos_pos - x_odd * sin_pos # Activations
        x_odd_rot = x_even * sin_pos + x_odd * cos_pos

        result_blocked = torch.stack((x_even_rot, x_odd_rot), dim=-1)
        result = result_blocked.reshape(*batch, seq_len, -1)

        return result


### Norm
rmsnorm_einx.py
**element-wise multiplies:**

**dot product:**

`0 matrix multiplies`

`0 FLOPs`

**trainable parameters:** `d_model`
* self.g(d_model)

In [ ]:
import torch
import torch.nn as nn
import einx

class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """ Construct the RMSNorm module.

        Args:
            d_model(int): Hidden dimension of the model
            eps(float): Epsilon value for numerical stability
            device(torch.device | None = None): Device to store the parameters on
            dtype(torch.dtype | None = None): Data type of the parameters
        """
        super().__init__()
        self.g = nn.Parameter(torch.ones(d_model, dtype=dtype, device=device))
        self.eps = eps
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Process an input tensor of shape

        Args:
            x (torch.Tensor): shape (batch_size, sequence_length, d_model)
        Returns:
            torch.Tensor: shape (batch_size, sequence_length, d_model)
        """
        in_dtype = x.dtype
        x = x.to(torch.float32)

        mean_square = einx.mean('... d -> ... 1', x * x)
        inv_rms = torch.rsqrt(mean_square + self.eps)

        x_norm = einx.multiply('... d, ... 1 -> ... d', x, inv_rms) # Activations: x(batch_size..., seq_len, d_model)
        result = einx.multiply('... d, d -> ... d', x_norm, self.g) # Activations: x_norm(batch_size..., seq_len, d_model), self.g[para-ignore](d_model)

        return result.to(in_dtype)


### Position-Wise Feed-Forward
positionwise_feedforward_einx.py

**element-wise multiplies:**

**dot product:**

`Num of matrix multiplies: 3 matrix multiplies`;

`FLOPs: (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs`
  * init: `0 matrix multiplies`, `0 FLOPs`
  * forward:
      * `3 matrix multiplies`
      * `(batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs`

**trainable parameters:** `3 * (d_ff * d_model)`
* init:
  * self.W1(d_ff, d_model)
  * self.W3(d_ff, d_model)
  * self.W2(d_model, d_ff)
* forward: 0 parameters

In [ ]:
import torch
import torch.nn as nn
import einx
from jaxtyping import Float
from torch import Tensor

class PWFFN(nn.Module):
    def __init__(self, d_model: int, d_ff: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()

        # d_ff = 8/3 * d_model

        self.W1 = nn.Parameter(torch.randn(d_ff, d_model, dtype=dtype, device=device)) # Parameter: self.W1(d_ff, d_model)
        self.W3 = nn.Parameter(torch.randn(d_ff, d_model, dtype=dtype, device=device)) # Parameter: self.W3(d_ff, d_model)

        self.W2 = nn.Parameter(torch.randn(d_model, d_ff, dtype=dtype, device=device)) # Parameter: self.W2(d_model, d_ff)
    def forward(self, x: Float[Tensor, " ... d_model"]) -> Float[Tensor, "... d_model"]:
        w1_item = einx.dot("... [d_model], [d_model] d_ff -> ... d_ff", x, self.W1.T) # x(... [d_model]), self.W1.T([d_model] d_ff); FLOPs = (batch...) * (2 * 1 * d_model * d_ff); # Activations: x(batch_size..., seq_len, d_model), self.W1.T[para-ignore](d_ff, d_model)
        w1_gate_item = PWFFN.silu(w1_item) # Parameter: 0 parameter; # Activations: w1_item(batch_size..., seq_len, d_ff)

        w3_item = einx.dot("... [d_model], [d_model] d_ff -> ... d_ff", x, self.W3.T) # x(... [d_model]), self.W3.T([d_model] d_ff); FLOPs = (batch...) * (2 * 1 * d_model * d_ff);

        l1 = einx.multiply("... d_ff, ... d_ff -> ... d_ff", w1_gate_item, w3_item)
        result = einx.dot("... [d_ff], [d_ff] d_model -> ... d_model", l1, self.W2.T) # l1(... [d_ff]), self.W2.T([d_ff] d_model); FLOPs = (batch...) * (2 * 1 * d_ff * d_model); # Activations: l1(batch_size..., seq_len, d_ff), self.W2.T[para-ignore](d_model, d_ff)

        return result

    @staticmethod
    def silu(x: Float[Tensor, "... d"]) -> Float[Tensor, "... d"]:
        return x * torch.sigmoid(x)

## Norm
rmsnorm_einx.py

** element-wise multiplies:**
** dot product:**

`Num of matrix multiplies: 0 matrix multiplies`

`FLOPs: 0 FLOPs`

**trainable parameters:** `d_model`
* self.g(d_model)


In [ ]:
import torch
import torch.nn as nn
import einx

class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """ Construct the RMSNorm module.

        Args:
            d_model(int): Hidden dimension of the model
            eps(float): Epsilon value for numerical stability
            device(torch.device | None = None): Device to store the parameters on
            dtype(torch.dtype | None = None): Data type of the parameters
        """
        super().__init__()
        self.g = nn.Parameter(torch.ones(d_model, dtype=dtype, device=device))
        self.eps = eps
    def forward(self, x: torch.Tensor) -> torch.Tensor: # Activations: x(batch_size..., seq_len, d_model)
        """Process an input tensor of shape

        Args:
            x (torch.Tensor): shape (batch_size, sequence_length, d_model)
        Returns:
            torch.Tensor: shape (batch_size, sequence_length, d_model)
        """
        in_dtype = x.dtype
        x = x.to(torch.float32)

        mean_square = einx.mean('... d -> ... 1', x * x)
        inv_rms = torch.rsqrt(mean_square + self.eps)

        x_norm = einx.multiply('... d, ... 1 -> ... d', x, inv_rms) # Activations: x(batch_size..., seq_len, d_model)
        result = einx.multiply('... d, d -> ... d', x_norm, self.g) # Activations: x_norm(batch_size..., seq_len, d_model), self.g[para-ignore](d_model)

        return result.to(in_dtype)


## Linear(Output Embedding)
linear_module.py

**element-wise multiplies:**
**dot product:**

`Num of matrix multiplies: 1 1 matrix multiplies`

`FLOPs: (batch...) * (2 * seq_len * d_model * vocab_size) FLOPs`
  * init: `0 matrix multiplies` `0 FLOPs`
  * forward:
      * `1 matrix multiplies`
      * `FLOPs = (batch...) * (2 * seq_len * d_model * vocab_size) FLOPs`

**trainable parameters:** `vocab_size * d_model`
* init:
  * self.W(vocab_size, d_model)
* forward: 0 parameters

In [ ]:
import torch
import torch.nn as nn
import math

class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """Construct a linear transformation module.

        Args:
            in_features(int): final dimension of the input
            out_features(int): final dimension of the output
            device(torch.device | None): Device to store the parameters on
            dtype(torch.dtype | None): Data type of the parameters
        """
        super().__init__()
        self.W = nn.Parameter(torch.randn(out_features, in_features, dtype=dtype, device=device)) # Parameter: self.W(vocab_size, d_model)
        std_variance = math.sqrt(2/(in_features + out_features))
        nn.init.trunc_normal_(self.W, mean=0, std=std_variance, a=-3*std_variance, b=3*std_variance)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the linear transformation to the input.
        """
        return x @ self.W.T # x(batch_size, seq_len, d_model), self.W.T(d_model, vocab_size); FLOPs = (batch...) * (2 * seq_len * d_model * vocab_size); # Activations: x(batch_size..., seq_len, d_model), self.W.T[para-ignore](vocab_size, d_model)

# Loss Function

In [ ]:
import torch
from jaxtyping import Float, Int
from torch import Tensor

def CrossEntropy(o: Float[Tensor, "batch_size vocab_size"], targets: Int[Tensor, " batch_size"]) -> Float[Tensor, ""]:
    target_expanded = targets.unsqueeze(-1)

    o_norm = o - o.max(dim=-1, keepdim=True).values

    o_target = o_norm.gather(dim=-1, index=target_expanded).squeeze(-1)
    # See cross_entropy_derivation.md for the full derivation.
    cross_entropy = torch.log(o_norm.exp().sum(-1)) - o_target
    return cross_entropy.mean() # Activations: o(batch_size..., seq_len, vocab_size)

# Optimizer(AdamW)

In [ ]:
from collections.abc import Callable
import torch
import math

class AdamW(torch.optim.Optimizer):
    def __init__(self, params, lr, betas, eps, weight_decay):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {
            "lr": lr,
            "betas": betas,
            "eps": eps,
            "weight_decay": weight_decay
        }
        super().__init__(params, defaults)
    def step(self, closure: Callable | None = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"]
            beta_1 = group["betas"][0]
            beta_2 = group["betas"][1]
            eps = group["eps"]
            weight_decay = group["weight_decay"]

            for p in group["params"]:
                if p.grad is None:
                    continue

                state = self.state[p]
                m = state.get("m", 0)
                v = state.get("v", 0)
                t = state.get("t", 1)

                grad = p.grad.data

                state["m"] = beta_1 * m + (1 - beta_1) * grad
                state["v"] = beta_2 * v + (1 - beta_2) * (grad ** 2)

                lr_t = lr * (math.sqrt(1 - math.pow(beta_2, t)) / (1 - math.pow(beta_1, t)))
                p.data = p.data - lr_t * (state["m"] / (torch.sqrt(state["v"]) + eps))
                p.data = p.data - lr * weight_decay * p.data

                state["t"] = t + 1

        return loss

# Q & A

Let us compute how much memory and compute running AdamW requires. Assume we are using float32 for every tensor.

**GPT-2 XL:**
* vocab_size : 50,257
* context_length : 1,024
* num_layers : 48
* d_model : 1,600
* num_heads : 25
* d_ff : 6,400

## (a)
How much peak memory does running AdamW require? Decompose your answer based on the memory usage of the parameters, activations, gradients, and optimizer state. Express your answer in terms of the batch_size and the model hyperparameters (`vocab_size`, `context_length`, `num_layers`, `d_model`, `num_heads`). Assume `d_ff` = 4 × `d_model`.

For simplicity, when calculating memory usage of activations, consider only the following components:
- Transformer block
  - RMSNorm(s)
  - Multi-head self-attention sublayer: $QKV$ projections, $Q^\top K$ matrix multiply, softmax, weighted sum of values, output projection.
  - Position-wise feed-forward: $W_1$ matrix multiply, SiLU, $W_2$ matrix multiply
- final RMSNorm
- output embedding
- cross-entropy on logits


In total(memory usage)

In [2]:
from sympy import symbols, simplify

# Declare symbolic variables
vocab_size, d_model, num_layers, d_ff, context_length, batch_size, num_heads = symbols(
    'vocab_size d_model num_layers d_ff context_length batch_size, num_heads'
)

# Define the expression
mem_expr = (
((
  num_layers *(
(
  4 * 2 * (
  batch_size * context_length * d_model
  + batch_size * context_length * d_model
))
+
(
  4 * batch_size * context_length * d_model
+ 4 * (batch_size * context_length * d_model + batch_size * num_heads * context_length * d_model)
+ 4 * (batch_size * num_heads * context_length * context_length)
+ 4 * (batch_size * num_heads * context_length * d_model)
+ 4 * (batch_size * context_length * d_model)
)
+
(
  4 * (batch_size * context_length * d_model)
+ 4 * (batch_size * context_length * d_ff)
+ 4 * (batch_size * context_length * d_ff)
)
)
)
+
4 * (
  batch_size * context_length * d_model
  + batch_size * context_length * d_model
)
+
4 * (batch_size * context_length * d_model)
+
4 * (batch_size * context_length * vocab_size)
)
+
(4 * (d_model*(2*num_layers*(8*d_model + 1) + 2*vocab_size + 1)))
+
(4 * (d_model*(2*num_layers*(8*d_model + 1) + 2*vocab_size + 1)))
+
(4 * (3*d_model*(2*num_layers*(8*d_model + 1) + 2*vocab_size + 1)))
)

# Substitute d_ff = 4 * d_model and Simplify the symbolic expression
mem_expr_expr_simplified = simplify(mem_expr.subs(d_ff, 4 * d_model))
print("Memory usage in bytes(expression):")
print(mem_expr_expr_simplified)
print()

Memory usage in bytes(expression):
12*batch_size*context_length*d_model + 4*batch_size*context_length*num_layers*(context_length*num_heads + 2*d_model*num_heads + 16*d_model) + 4*batch_size*context_length*vocab_size + 20*d_model*(2*num_layers*(8*d_model + 1) + 2*vocab_size + 1)



### parameters

num

In [ ]:
from sympy import symbols, simplify

# Declare symbolic variables
vocab_size, d_model, num_layers, d_ff = symbols(
    'vocab_size d_model num_layers d_ff'
)

# Define the expression
# vocab_size * d_model + num_layers * (d_model + d_model + 4 * (d_model * d_model) + 3 * (d_ff * d_model)) + d_model + vocab_size * d_model
num_para_expr = (
    vocab_size * d_model
    + num_layers * (d_model + d_model + 4 * (d_model * d_model) + 3 * (d_ff * d_model))
    + d_model
    + vocab_size * d_model
)

# Substitute d_ff = 4 * d_model and Simplify the symbolic expression
num_para_expr_simplified = simplify(num_para_expr.subs(d_ff, 4 * d_model))
print("Num of Parameters(expression):")
print(num_para_expr_simplified)
print()


Num of Parameters(Simplified symbolic expression):
d_model*(2*num_layers*(8*d_model + 1) + 2*vocab_size + 1)



memory usage

```
4 * (d_model*(2*num_layers*(8*d_model + 1) + 2*vocab_size + 1)) bytes
```


### gradients

num

In [ ]:
print(f"Num of Gradients(expression): {num_para_expr_simplified}")

Num of Gradients(expression): d_model*(2*num_layers*(8*d_model + 1) + 2*vocab_size + 1)


memory usage

```
4 * (d_model*(2*num_layers*(8*d_model + 1) + 2*vocab_size + 1))
```

### optimizer state

num

In [ ]:
print(f"Num of m(first moments): {num_para_expr_simplified}")

print(f"Num of v(second moments): {num_para_expr_simplified}")

print(f"Num of t(iteration t): {num_para_expr_simplified}")

print(f"Num of total optimizer state: {simplify(3 * num_para_expr_simplified)}")

Num of m(first moments): d_model*(2*num_layers*(8*d_model + 1) + 2*vocab_size + 1)
Num of v(second moments): d_model*(2*num_layers*(8*d_model + 1) + 2*vocab_size + 1)
Num of t(iteration t): d_model*(2*num_layers*(8*d_model + 1) + 2*vocab_size + 1)
Num of total optimizer state: 3*d_model*(2*num_layers*(8*d_model + 1) + 2*vocab_size + 1)


memory usage

```
4 * (3*d_model*(2*num_layers*(8*d_model + 1) + 2*vocab_size + 1))
```

### activations

memory usage of activations:

```
(
  num_layers *(
(
  4 * 2 * (
  batch_size * sequence_length * d_model
  + batch_size * seq_len * d_model
))
+
(
  4 * batch_size * seq_len * d_model
+ 4 * (batch_size * seq_len * d_model + batch_size * num_heads * seq_len * d_model)
+ 4 * (batch_size * num_heads * seq_len * seq_len)
+ 4 * (batch_size * num_heads * seq_len * d_model)
+ 4 * (batch_size * seq_len * d_model)
)
+
(
  4 * (batch_size * seq_len * d_model)
+ 4 * (batch_size * seq_len * d_ff)
+ 4 * (batch_size * seq_len * d_ff)
)
)
)
+
4 * (
  batch_size * sequence_length * d_model
  + batch_size * seq_len * d_model
)
+
4 * (batch_size * seq_len * d_model)
+
4 * (batch_size * seq_len * vocab_size)
```

#### Transformer block

memory usage of Transformer block:

```
num_layers * (
(
  4 * 2 * (
  batch_size * sequence_length * d_model
  + batch_size * seq_len * d_model
))
+
(
  4 * batch_size * seq_len * d_model
+ 4 * (batch_size... * seq_len * d_model + batch_size * num_heads * seq_len * d_model)
+ 4 * (batch_size * num_heads * seq_len * seq_len)
+ 4 * (batch_size * num_heads * seq_len * d_model)
+ 4 * (batch_size * seq_len * d_model)
)
+
(
  4 * (batch_size * seq_len * d_model)
+ 4 * (batch_size * seq_len * d_ff)
+ 4 * (batch_size * seq_len * d_ff)
)
)
```

##### RMSNorm(s)

num:

```
batch_size * sequence_length * d_model
+ batch_size * seq_len * d_model
```

memory usage:
```
4 * 2 * (
  batch_size * sequence_length * d_model
  + batch_size * seq_len * d_model
)
```

##### Multi-head self-attention sublayer

memory usage of Multi-head self-attention sublayer:

```
4 * batch_size * seq_len * d_model
+ 4 * (batch_size... * seq_len * d_model + batch_size * num_heads * seq_len * d_model)
+ 4 * (batch_size * num_heads * seq_len * seq_len)
+ 4 * (batch_size * num_heads * seq_len * d_model)
+ 4 * (batch_size * seq_len * d_model)
```

###### $QKV$ projections

num:

`batch_size * seq_len * d_model`

memory usage:

```
4 * batch_size * seq_len * d_model
```

###### $Q^\top K$ matrix multiply

num:

```
batch_size... * seq_len * d_model
+ batch_size * num_heads * seq_len * d_model
```

memory usage:

```
4 * (
  batch_size... * seq_len * d_model
  + batch_size * num_heads * seq_len * d_model
)
```

###### softmax

num:

`batch_size * num_heads * seq_len * seq_len`

memory usage:

```
4 * (batch_size * num_heads * seq_len * seq_len)
```

###### weighted sum of values

num:

`batch_size * num_heads * seq_len * d_model`

memory usage:
```
4 * (batch_size * num_heads * seq_len * d_model)
```

###### output projection

num:

`batch_size * seq_len * d_model`

memory usage:

```
4 * (batch_size * seq_len * d_model)
```

##### Position-wise feed-forward

memory usage of Position-wise feed-forward:

```
4 * (batch_size * seq_len * d_model)
+ 4 * (batch_size * seq_len * d_ff)
+ 4 * (batch_size * seq_len * d_ff)
```

###### $W_1$ matrix multiply

num:

`batch_size * seq_len * d_model`

memory usage:

```
4 * (batch_size * seq_len * d_model)
```

###### SiLU

num:

`batch_size * seq_len * d_ff`

memory usage:

```
4 * (batch_size * seq_len * d_ff)
```

###### $W_2$ matrix multiply

num:

`batch_size * seq_len * d_ff`

memory usage:

```
4 * (batch_size * seq_len * d_ff)
```

#### final RMSNorm

num:

```
batch_size * sequence_length * d_model
+ batch_size * seq_len * d_model
```

memory usage:

```
4 * (
  batch_size * sequence_length * d_model
  + batch_size * seq_len * d_model
)
```

#### output embedding

num:

```
batch_size * seq_len * d_model
```

memory usage:

```
4 * (batch_size * seq_len * d_model)
```

#### cross-entropy on logits

num:

```
batch_size * seq_len * vocab_size
```

memory usage:

```
4 * (batch_size * seq_len * vocab_size)